# Assessment 2: Machine Learning and Real-Time Streaming

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In Assessment 1, we analysed historical freight charges across Olist shipping lanes. Now, in this task we want to build on that work using the same dataset. Firstly, in Part A, we look to train a model to predict the freight charged on each order item. Then, later in Part B, we want to be able to apply the saved model to orders it has not seen.

---

## Contents

**Setup**
- Environment and configuration
- Data loading
- Building the modelling dataset
- Train/stream split

**Part A: Machine learning pipeline**
1. ML task selection
2. Feature engineering
3. Model comparison and evaluation
4. Model persistence and verification

**Part B: Streaming, Kafka and reflection**
1. Streaming data simulation and Kafka producer
2. Structured Streaming consumer
3. Windowed aggregation
4. Reflection

---

## Environment and configuration

### Execution environment

Similar to what we did previously in Assessment 1, we want to run Spark in local mode on a single laptop, with its four logical cores acting as parallel task slots. The key difference for us now in this assessment is that a Kafka broker and ZooKeeper also run on the same machine during Part B. 

As we note in the below table, all three share the 8GB of memory. As such, Kafka's memory is capped at 512MB in order to leave enough room for Spark.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Kafka | 3.9.2 (ZooKeeper mode) |
| Spark master | `local[*]` |

In [2]:
# Import packages
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# File locations
DATA_DIR = "data/raw"
STREAM_PATH = "data/stream_data.parquet"
MODEL_PATH = "models/a2_model"

# Random seed for the train/stream split + model training
SEED = 42

### SparkSession configuration

In order for us to be able to complete the tasks needed for this assessment within our local set up, there were a few key Spark settings that we first made changes to. These are:

**`spark.driver.memory = 3g`**

Given that our notebook is running Spark in local mode, the driver is therefore doing the processing on the laptop itself. As such, we wanted to give Spark 3GB of memory rather than letting allowing for it to consume too much memory. 
This decision was made due to the fact that the laptop has 8 GB in available memory, and our later tasks in Part B need Kafka and ZooKeeper running alongside Spark. Because of this, leaving some memory free for those processes was critical.

**`spark.sql.shuffle.partitions = 4`**

Similar to our decision in Assessment 1, this setting was again set to 4. We note that the normal Spark default is 200 partitions, which is excessive for our dataset, which only has ~110,000 rows and would create lots of tiny tasks unneccesarily.

Furthermore, this decision also matters for our streaming section of this assessment. The shuffle partition setting is used by Spark when maintaining aggregation state, so using 200 partitions would mean each micro-batch has much more scheduling overhead. On the other hand, four partitions is more appropriate for the small local setup being used here.

**`spark.sql.session.timeZone = UTC`**

UTC is used here so that the timestamp handling stays consistent between runs rather than depending on the laptop's local time settings. This is particularly useful in Part B because the produced Kafka records contain timestamps.

It also avoids issues caused by Sydney changing between standard time and daylight saving time.

**`spark.jars.packages`**

We needed to add the Kafka connector separately due to the fact that it is not included in the normal PySpark installation. The connector version is matched to the Spark/PySpark version being used.

**Adaptive Query Execution**

AQE has not been manually disabled for this assessment, so Spark keeps its normal default behaviour.

In our previous assessment, AQE was turned off because the physical execution plans were being examined and needed to stay predictable. Given that this is not required here, there was no particular need for us to disable it. We note that AQE also does not apply to the streaming query itself, meaning it is only relevant to our batch work in Part A.

In [3]:
# Start building the Spark session
builder = SparkSession.builder

# Basic session settings
builder = builder.appName("ITO5202-A2-Olist-Freight")
builder = builder.master("local[*]")

# Memory and partition settings
builder = builder.config("spark.driver.memory", "3g")
builder = builder.config("spark.sql.shuffle.partitions", 4)

# Keep timestamps consistent
builder = builder.config("spark.sql.session.timeZone", "UTC")

# Add the Kafka connector
builder = builder.config(
    "spark.jars.packages",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
)

# Create the session
spark = builder.getOrCreate()

# Reduce console logging
spark.sparkContext.setLogLevel("ERROR")

spark

26/10/02 18:46:11 WARN Utils: Your hostname, Mounishas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.137 instead (on interface en0)
26/10/02 18:46:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/mmondeddu9/Desktop/ito5202-assessment2-streaming/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/mmondeddu9/.ivy2/cache
The jars for the packages stored in: /Users/mmondeddu9/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1de4b396-a890-4241-91e3-687f7ea492e9;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1248ms :: artifacts 

In [4]:
# Get the Spark context
sc = spark.sparkContext

# Check the main environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Task slots:", sc.defaultParallelism)

# Check the configured values
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Time zone:", spark.conf.get("spark.sql.session.timeZone"))
print("Kafka package:", spark.conf.get("spark.jars.packages"))

Spark version: 3.5.1
Master: local[*]
Task slots: 4
Driver memory: 3g
Shuffle partitions: 4
AQE: true
Time zone: UTC
Kafka package: org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1


---

## Data loading

### Schemas

The seven CSV files that we need for our analysis are loaded with schemas written out manually instead of using `inferSchema=True`.

We note that it is particularly important for us to do this due to two key reasons:

1. Spark needs to inspect the data first when it infers a schema, which means another pass over the files before the actual load. This is unneccesary (and inefficient) for our use case, especially for the geolocation file which has over one million rows.

2. By using fixed schemas, we are also able to make the data types more predictable. Columns used across multiple tables, such as `order_id` and `product_id`, are defined the same way each time, which helps us to avoid type problems later on during joins.

We should also note again that the payments and reviews files are not used in this project. This is due to the fact that reviews only exist after an order has been delivered, and as such they would not be known, or relevant when freight is calculated. Also, payment values are also not used because they already include freight as part of the total amount paid. We consider these exclusions and our reasoning in more detail later in Part A.

### Correcting source column names

We note that there are two spelling mistakes in the original products CSV headers: `product_name_lenght` and `product_description_lenght`.

The initial schema still needs to use those exact names so Spark can read the file correctly. After the file is loaded, we can correct these column names.

In [5]:
# Schemas for the seven CSV files
order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])

category_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True)
])


# Small helper for loading CSV files
def load_csv(filename, schema):
    file_path = DATA_DIR + "/" + filename

    df = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv(file_path)

    return df


# Load each dataset
order_items = load_csv("olist_order_items_dataset.csv", order_items_schema)
orders = load_csv("olist_orders_dataset.csv", orders_schema)
customers = load_csv("olist_customers_dataset.csv", customers_schema)
sellers = load_csv("olist_sellers_dataset.csv", sellers_schema)
geolocation = load_csv("olist_geolocation_dataset.csv", geolocation_schema)
categories = load_csv("product_category_name_translation.csv", category_schema)

# Load products separately so the headers can be fixed
products = load_csv("olist_products_dataset.csv", products_schema)

# Correct the two spelling mistakes
products = products.withColumnRenamed(
    "product_name_lenght",
    "product_name_length"
)

products = products.withColumnRenamed(
    "product_description_lenght",
    "product_description_length"
)

In [6]:
# Store each dataset with its expected row count
datasets = [
    ("order_items", order_items, 112650),
    ("orders", orders, 99441),
    ("customers", customers, 99441),
    ("sellers", sellers, 3095),
    ("geolocation", geolocation, 1000163),
    ("products", products, 32951),
    ("categories", categories, 71)
]

# Build a simple validation table
rows = []

for name, df, expected in datasets:
    actual = df.count()
    number_of_columns = len(df.columns)

    row = {
        "dataset": name,
        "expected": expected,
        "actual": actual,
        "match": actual == expected,
        "columns": number_of_columns
    }

    rows.append(row)

# Display the results
count_check = pd.DataFrame(rows)

count_check

,dataset,expected,actual,match,columns
0,order_items,112650,112650,True,7
1,orders,99441,99441,True,8
2,customers,99441,99441,True,5
3,sellers,3095,3095,True,4
4,geolocation,1000163,1000163,True,5
5,products,32951,32951,True,9
6,categories,71,71,True,2
